# Stage 2 V3: Training depuis Stage 1 Checkpoint

## Analyse des versions précédentes

| Version | Accuracy | Problème |
|---------|----------|----------|
| Stage 1 | **80.52%** | Classification uniquement |
| Stage 2 V1 | 45% | Catastrophic forgetting (CMD 95%, autres ~0%) |
| Stage 2 V2 | 31% | Recommencé depuis base → perte capacités |

## Solution V3

**Partir du checkpoint Stage 1** (80.52%) et fine-tuner avec:
- LR ultra-bas: **5e-7** (40x plus bas que V1)
- Replay buffer: **30%** (vs 15% en V2)
- 1 epoch (conservateur)
- LoRA existant en mode trainable

## Objectifs V3
- Global: ≥70% (compromis classification + VQA)
- Maintenir les capacités Stage 1 autant que possible

In [1]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================

import os
import json
import re
import random
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from dataclasses import dataclass
from typing import Dict, List, Any, Optional
from collections import defaultdict, Counter
from datasets import Dataset
from tqdm import tqdm

# IMPORTANT: Import unsloth AVANT transformers
from unsloth import FastVisionModel
from peft import PeftModel

from transformers import (
    AutoProcessor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.9.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
GPU Memory: 33.67 GB


In [2]:
# ============================================================
# CELL 2: CONFIGURATION V3
# ============================================================

# Chemins
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
STAGE1_CHECKPOINT = BASE_DIR / "checkpoints" / "stage1_cassava_qwen25vl_7b" / "final_model"
STAGE1_DATA_DIR = BASE_DIR / "data" / "stage1"
STAGE2_DATA_PATH = BASE_DIR / "data" / "stage2" / "stage2_vqa_dataset.json"
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "stage2_v3_from_stage1"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Modèle de base (même que Stage 1)
MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)

# ============================================================
# CONFIGURATION V3 - ULTRA CONSERVATEUR
# ============================================================

# Training Configuration
TRAINING_CONFIG = {
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,      # Effective batch = 16
    "num_train_epochs": 1,                 # 1 SEUL epoch (conservateur)
    "learning_rate": 5e-7,                 # ULTRA BAS (40x moins que V1)
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "max_grad_norm": 0.3,                  # Gradient clipping agressif
    "logging_steps": 50,
    "save_steps": 500,
    "eval_steps": 250,
    "bf16": torch.cuda.is_bf16_supported(),
    "fp16": not torch.cuda.is_bf16_supported(),
}

# Replay buffer plus important
REPLAY_CONFIG = {
    "n_samples": 600,           # 600 samples Stage 1 (doublé)
    "mix_ratio": 0.30,          # 30% du dataset final
}

# Pas de rééquilibrage agressif - garder distribution naturelle
BALANCE_CONFIG = {
    "target_per_class": 5000,   # Plus modeste
}

# Classes
CLASS_NAMES = ["CBB", "CBSD", "CGM", "CMD", "Healthy"]
CLASS_FULL_NAMES = {
    "CBB": "Bactériose du manioc (CBB)",
    "CBSD": "Striure brune du manioc (CBSD)",
    "CGM": "Acarien vert du manioc (CGM)",
    "CMD": "Mosaïque du manioc (CMD)",
    "Healthy": "Sain"
}

# Stage 1 results for comparison
STAGE1_RESULTS = {
    "global": 80.52,
    "CBB": 55.13,
    "CBSD": 83.99,
    "CGM": 56.47,
    "CMD": 85.98,
    "Healthy": 75.37
}

print(f"{'='*60}")
print(f"CONFIGURATION STAGE 2 V3 - FROM STAGE 1")
print(f"{'='*60}")
print(f"\n📊 Différences clés vs V2:")
print(f"  - Base: Stage 1 checkpoint (80.52%) vs modèle de base")
print(f"  - LR: {TRAINING_CONFIG['learning_rate']} (vs 2e-6 en V2)")
print(f"  - Epochs: {TRAINING_CONFIG['num_train_epochs']} (vs 2 en V2)")
print(f"  - Replay: {REPLAY_CONFIG['n_samples']} samples ({REPLAY_CONFIG['mix_ratio']*100}%)")
print(f"  - Grad clipping: {TRAINING_CONFIG['max_grad_norm']} (agressif)")
print(f"{'='*60}")

CONFIGURATION STAGE 2 V3 - FROM STAGE 1

📊 Différences clés vs V2:
  - Base: Stage 1 checkpoint (80.52%) vs modèle de base
  - LR: 5e-07 (vs 2e-6 en V2)
  - Epochs: 1 (vs 2 en V2)
  - Replay: 600 samples (30.0%)
  - Grad clipping: 0.3 (agressif)


In [3]:
# ============================================================
# CELL 3: CHARGEMENT STAGE 1 CHECKPOINT
# ============================================================

print("\n" + "="*60)
print("LOADING STAGE 1 CHECKPOINT")
print("="*60)

print(f"\n1. Loading base model...")
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

print(f"\n2. Loading Stage 1 LoRA weights...")
print(f"   Checkpoint: {STAGE1_CHECKPOINT}")

# Charger les poids LoRA Stage 1 en mode trainable
model = PeftModel.from_pretrained(
    model, 
    str(STAGE1_CHECKPOINT),
    is_trainable=True  # IMPORTANT: permet de continuer l'entraînement
)

print(f"\n3. Loading processor...")
processor = AutoProcessor.from_pretrained(str(STAGE1_CHECKPOINT))

# Statistiques
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"\n✅ Stage 1 checkpoint chargé")
print(f"   - Stage 1 accuracy: {STAGE1_RESULTS['global']:.2f}%")
print(f"   - Trainable params: {trainable:,} ({100*trainable/total:.4f}%)")
print(f"   - Mode: is_trainable=True (fine-tuning continu)")
print("="*60)


LOADING STAGE 1 CHECKPOINT

1. Loading base model...
==((====))==  Unsloth 2026.1.1: Fast Qwen2_5_Vl patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

2. Loading Stage 1 LoRA weights...
   Checkpoint: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_qwen25vl_7b/final_model

3. Loading processor...

✅ Stage 1 checkpoint chargé
   - Stage 1 accuracy: 80.52%
   - Trainable params: 2,523,136 (0.0501%)
   - Mode: is_trainable=True (fine-tuning continu)


In [4]:
# ============================================================
# CELL 4: CRÉATION REPLAY BUFFER (30%)
# ============================================================

print("\n" + "="*60)
print("CREATING REPLAY BUFFER (30%)")
print("="*60)

def load_stage1_data():
    """Charge les données Stage 1."""
    train_path = STAGE1_DATA_DIR / "cassava_train_balanced.json"
    with open(train_path, 'r') as f:
        return json.load(f)

def extract_image_path(text):
    """Extrait le chemin de l'image."""
    match = re.search(r'<img>(.*?)</img>', text)
    return match.group(1) if match else None

def get_class_from_response(response: str) -> str:
    """Extrait la classe de la réponse Stage 1."""
    response_lower = response.lower()
    if "cbb" in response_lower or "bacterial blight" in response_lower:
        return "CBB"
    elif "cbsd" in response_lower or "brown streak" in response_lower:
        return "CBSD"
    elif "cgm" in response_lower or "green mottle" in response_lower or "green mite" in response_lower:
        return "CGM"
    elif "cmd" in response_lower or "mosaic" in response_lower:
        return "CMD"
    elif "healthy" in response_lower:
        return "Healthy"
    return "Unknown"

def create_replay_buffer(stage1_samples, n_samples=600):
    """
    Crée un replay buffer équilibré depuis les données Stage 1.
    Garde le format ORIGINAL Stage 1 (pas de conversion VQA).
    """
    
    # Organiser par classe
    samples_by_class = defaultdict(list)
    for s in stage1_samples:
        for turn in s['conversations']:
            if turn['from'] == 'assistant':
                cls = get_class_from_response(turn['value'])
                if cls != "Unknown":
                    samples_by_class[cls].append(s)
                break
    
    # Sélectionner équilibré par classe
    per_class = n_samples // 5
    selected = []
    
    for cls in CLASS_NAMES:
        cls_samples = samples_by_class[cls]
        if len(cls_samples) >= per_class:
            selected.extend(random.sample(cls_samples, per_class))
        else:
            # Oversample si pas assez
            multiplier = (per_class // len(cls_samples)) + 1
            oversampled = (cls_samples * multiplier)[:per_class]
            selected.extend(oversampled)
    
    # Ajouter metadata pour tracking
    replay_samples = []
    for s in selected:
        # Extraire la classe
        cls = "Unknown"
        for turn in s['conversations']:
            if turn['from'] == 'assistant':
                cls = get_class_from_response(turn['value'])
                break
        
        replay_sample = {
            'id': f"replay_{s['id']}",
            'conversations': s['conversations'],
            'metadata': {
                'source': 'replay_stage1',
                'class': cls,
                'severity': 'general',
                'qa_type': 'identification'
            }
        }
        replay_samples.append(replay_sample)
    
    return replay_samples

# Charger Stage 1 et créer replay buffer
stage1_data = load_stage1_data()
print(f"Stage 1 data loaded: {len(stage1_data)} samples")

replay_buffer = create_replay_buffer(stage1_data, n_samples=REPLAY_CONFIG['n_samples'])
print(f"\n✅ Replay buffer créé: {len(replay_buffer)} samples")

# Vérifier distribution
replay_dist = Counter(s['metadata']['class'] for s in replay_buffer)
print("\n📊 Distribution replay buffer:")
for cls in CLASS_NAMES:
    print(f"  {cls}: {replay_dist[cls]}")


CREATING REPLAY BUFFER (30%)
Stage 1 data loaded: 29276 samples

✅ Replay buffer créé: 600 samples

📊 Distribution replay buffer:
  CBB: 120
  CBSD: 120
  CGM: 120
  CMD: 120
  Healthy: 120


In [5]:
# ============================================================
# CELL 5: CHARGEMENT ET MIXAGE STAGE 2 + REPLAY
# ============================================================

print("\n" + "="*60)
print("LOADING AND MIXING DATASETS")
print("="*60)

# Charger Stage 2
with open(STAGE2_DATA_PATH, 'r', encoding='utf-8') as f:
    stage2_data = json.load(f)

print(f"Stage 2 data loaded: {len(stage2_data)} samples")

# Analyser distribution Stage 2
stage2_dist = Counter(s['metadata']['class'] for s in stage2_data)
print("\n📊 Distribution Stage 2 originale:")
for cls, count in stage2_dist.most_common():
    print(f"  {cls}: {count} ({count/len(stage2_data)*100:.1f}%)")

# Rééquilibrage modéré (moins agressif que V2)
def balance_dataset_moderate(data, target_per_class=5000):
    """Rééquilibre modérément le dataset."""
    samples_by_class = defaultdict(list)
    for s in data:
        cls = s['metadata']['class']
        samples_by_class[cls].append(s)
    
    balanced = []
    for cls in CLASS_NAMES:
        cls_samples = samples_by_class[cls]
        n = len(cls_samples)
        
        if n >= target_per_class:
            balanced.extend(random.sample(cls_samples, target_per_class))
        else:
            # Oversampling modéré (max 3x)
            multiplier = min(3, (target_per_class // n) + 1)
            oversampled = (cls_samples * multiplier)[:target_per_class]
            balanced.extend(oversampled)
    
    random.shuffle(balanced)
    return balanced

print("\n🔄 Rééquilibrage modéré...")
stage2_balanced = balance_dataset_moderate(stage2_data, target_per_class=BALANCE_CONFIG['target_per_class'])
print(f"Stage 2 après rééquilibrage: {len(stage2_balanced)} samples")

# Mixer Stage 2 + Replay
print("\n🔀 Mixage Stage 2 + Replay buffer...")
mixed_data = stage2_balanced + replay_buffer
random.shuffle(mixed_data)

# Statistiques
total = len(mixed_data)
replay_pct = len(replay_buffer) / total * 100

print(f"\n📊 Dataset mixé:")
print(f"  - Stage 2 VQA: {len(stage2_balanced)} ({100-replay_pct:.1f}%)")
print(f"  - Replay Stage 1: {len(replay_buffer)} ({replay_pct:.1f}%)")
print(f"  - Total: {total}")

# Distribution finale
final_dist = Counter(s['metadata']['class'] for s in mixed_data)
print("\n📊 Distribution finale:")
for cls in CLASS_NAMES:
    print(f"  {cls}: {final_dist[cls]} ({final_dist[cls]/total*100:.1f}%)")


LOADING AND MIXING DATASETS
Stage 2 data loaded: 29276 samples

📊 Distribution Stage 2 originale:
  CMD: 12000 (41.0%)
  CBSD: 5296 (18.1%)
  CGM: 5054 (17.3%)
  CBB: 3717 (12.7%)
  Healthy: 3209 (11.0%)

🔄 Rééquilibrage modéré...
Stage 2 après rééquilibrage: 25000 samples

🔀 Mixage Stage 2 + Replay buffer...

📊 Dataset mixé:
  - Stage 2 VQA: 25000 (97.7%)
  - Replay Stage 1: 600 (2.3%)
  - Total: 25600

📊 Distribution finale:
  CBB: 5120 (20.0%)
  CBSD: 5120 (20.0%)
  CGM: 5120 (20.0%)
  CMD: 5120 (20.0%)
  Healthy: 5120 (20.0%)


In [6]:
# ============================================================
# CELL 6: PRÉPARATION DATASET ET DATA COLLATOR
# ============================================================

def format_sample_for_training(sample):
    """Formate un sample pour l'entraînement."""
    conversations = sample['conversations']
    
    image_path = None
    user_text = ""
    assistant_text = ""
    
    for turn in conversations:
        if turn['from'] == 'user':
            match = re.search(r'<img>(.*?)</img>', turn['value'])
            if match:
                image_path = match.group(1)
            user_text = re.sub(r'Picture \d+: <img>.*?</img>\n?', '', turn['value']).strip()
        elif turn['from'] == 'assistant':
            assistant_text = turn['value']
    
    return {
        "conversations": [
            {"role": "user", "content": f"Picture 1: <image>\n{user_text}"},
            {"role": "assistant", "content": assistant_text}
        ],
        "image_path": image_path,
        "id": sample['id'],
        "class_name": sample['metadata']['class'],
        "source": sample['metadata'].get('source', 'stage2')
    }

print("\nFormatting data...")
all_formatted = [format_sample_for_training(s) for s in tqdm(mixed_data, desc="Formatting")]

# Filtrer images valides
valid_samples = [s for s in all_formatted if s['image_path'] and os.path.exists(s['image_path'])]
print(f"\n✅ Valid samples: {len(valid_samples)} / {len(all_formatted)}")

# Split train/val (95/5 pour garder plus de données d'entraînement)
random.seed(42)
random.shuffle(valid_samples)
split_idx = int(len(valid_samples) * 0.95)
train_formatted = valid_samples[:split_idx]
val_formatted = valid_samples[split_idx:]

print(f"Train: {len(train_formatted)}")
print(f"Val: {len(val_formatted)}")

# Créer datasets
train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(val_formatted)


Formatting data...


Formatting: 100%|██████████| 25600/25600 [00:00<00:00, 418931.28it/s]


✅ Valid samples: 25600 / 25600
Train: 24320
Val: 1280


In [7]:
# ============================================================
# CELL 7: DATA COLLATOR
# ============================================================

@dataclass
class QwenDataCollator:
    """Data collator pour Qwen2.5-VL."""
    processor: Any
    max_length: int = 1536

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_messages = []
        batch_images = []

        for feature in features:
            image_path = feature['image_path']
            image = None

            if image_path and os.path.exists(image_path):
                try:
                    image = Image.open(image_path).convert('RGB')
                    image = image.resize(TARGET_IMAGE_SIZE, resample=Image.BILINEAR)
                except:
                    image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')
            else:
                image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')

            batch_images.append([image])

            conversations = feature['conversations']
            messages = []
            
            for turn in conversations:
                role = turn['role']
                content = turn['content']

                if role == 'user' and len(messages) == 0:
                    text_content = content.replace('<image>', '').replace('Picture 1:', '').strip()
                    messages.append({
                        "role": role,
                        "content": [
                            {"type": "image", "image": image},
                            {"type": "text", "text": text_content}
                        ]
                    })
                else:
                    messages.append({
                        "role": role,
                        "content": content.replace('<image>', '').strip()
                    })

            batch_messages.append(messages)

        batch_texts_full = []
        batch_texts_user = []

        for messages in batch_messages:
            full_text = self.processor.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            batch_texts_full.append(full_text)

            user_messages = [msg for msg in messages if msg["role"] == "user"]
            user_text = self.processor.tokenizer.apply_chat_template(
                user_messages, tokenize=False, add_generation_prompt=True
            )
            batch_texts_user.append(user_text)

        batch = self.processor(
            text=batch_texts_full,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length
        )

        labels = batch["input_ids"].clone()

        for i in range(len(batch_texts_user)):
            user_encoding = self.processor(
                text=[batch_texts_user[i]],
                images=[batch_images[i]],
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=self.max_length
            )
            user_length = user_encoding["input_ids"].shape[1]

            if user_length < labels[i].shape[0]:
                labels[i, :user_length] = -100
            else:
                labels[i, :-20] = -100

        batch["labels"] = labels
        return batch

data_collator = QwenDataCollator(processor=processor)
print("✅ Data collator créé")

✅ Data collator créé


In [8]:
# ============================================================
# CELL 8: TRAINER V3
# ============================================================

class Stage2V3Trainer(Trainer):
    """Trainer pour Stage 2 V3."""
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        return (loss, outputs) if return_outputs else loss

print("\n" + "="*60)
print("CONFIGURING TRAINER V3")
print("="*60)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
    lr_scheduler_type="cosine",
    
    weight_decay=TRAINING_CONFIG["weight_decay"],
    max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
    
    logging_steps=TRAINING_CONFIG["logging_steps"],
    save_steps=TRAINING_CONFIG["save_steps"],
    save_total_limit=3,
    
    eval_strategy="steps",
    eval_steps=TRAINING_CONFIG["eval_steps"],
    
    bf16=TRAINING_CONFIG["bf16"],
    fp16=TRAINING_CONFIG["fp16"],
    
    optim="adamw_8bit",
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    seed=42,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=5,
    early_stopping_threshold=0.001
)

trainer = Stage2V3Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[early_stopping],
)

effective_batch = TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']
total_steps = len(train_dataset) // effective_batch * TRAINING_CONFIG['num_train_epochs']

print(f"\n📊 Configuration V3:")
print(f"  - Base: Stage 1 checkpoint (80.52% accuracy)")
print(f"  - Train samples: {len(train_dataset):,}")
print(f"  - Effective batch: {effective_batch}")
print(f"  - Total steps: ~{total_steps:,}")
print(f"  - LR: {TRAINING_CONFIG['learning_rate']} (ULTRA BAS)")
print(f"  - Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"  - Grad clipping: {TRAINING_CONFIG['max_grad_norm']}")
print("="*60)


CONFIGURING TRAINER V3

📊 Configuration V3:
  - Base: Stage 1 checkpoint (80.52% accuracy)
  - Train samples: 24,320
  - Effective batch: 16
  - Total steps: ~1,520
  - LR: 5e-07 (ULTRA BAS)
  - Epochs: 1
  - Grad clipping: 0.3


In [9]:
# ============================================================
# CELL 9: ENTRAÎNEMENT V3
# ============================================================

print("\n" + "="*60)
print("STARTING STAGE 2 V3 TRAINING")
print("="*60)
print(f"\n🎯 Stratégie V3:")
print(f"  1. Base: Stage 1 checkpoint (80.52%)")
print(f"  2. LR ultra-bas: {TRAINING_CONFIG['learning_rate']}")
print(f"  3. Replay buffer: {len(replay_buffer)} samples ({REPLAY_CONFIG['mix_ratio']*100}%)")
print(f"  4. 1 seul epoch (conservateur)")
print(f"\n🎯 Objectif: ≥70% global avec capacités VQA")
print("="*60 + "\n")

train_result = trainer.train()

print("\n" + "="*60)
print("STAGE 2 V3 TRAINING COMPLETED")
print("="*60)
print(f"Final loss: {train_result.training_loss:.4f}")
print(f"Total steps: {train_result.global_step}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24,320 | Num Epochs = 1 | Total steps = 1,520
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 2,523,136 of 8,294,689,792 (0.03% trained)



STARTING STAGE 2 V3 TRAINING

🎯 Stratégie V3:
  1. Base: Stage 1 checkpoint (80.52%)
  2. LR ultra-bas: 5e-07
  3. Replay buffer: 600 samples (30.0%)
  4. 1 seul epoch (conservateur)

🎯 Objectif: ≥70% global avec capacités VQA

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
250,31.592500,3.842703
500,30.537900,3.796985
750,31.390000,3.763113
1000,30.515800,3.737822
1250,29.734500,3.727125
1500,29.635700,3.725849



STAGE 2 V3 TRAINING COMPLETED
Final loss: 30.5159
Total steps: 1520


In [10]:
# ============================================================
# CELL 10: SAUVEGARDE
# ============================================================

print("\nSaving Stage 2 V3 model...")
final_model_path = CHECKPOINT_DIR / "final_model"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

config = {
    "stage": "2_v3_from_stage1",
    "base_checkpoint": str(STAGE1_CHECKPOINT),
    "stage1_accuracy": STAGE1_RESULTS['global'],
    "training_config": TRAINING_CONFIG,
    "replay_config": REPLAY_CONFIG,
    "balance_config": BALANCE_CONFIG,
    "training_loss": train_result.training_loss,
    "total_steps": train_result.global_step,
    "train_samples": len(train_dataset),
    "replay_samples": len(replay_buffer),
}

with open(CHECKPOINT_DIR / "training_config.json", 'w') as f:
    json.dump(config, f, indent=2)

print(f"✅ Stage 2 V3 model saved to: {final_model_path}")


Saving Stage 2 V3 model...
✅ Stage 2 V3 model saved to: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v3_from_stage1/final_model


In [11]:
# ============================================================
# CELL 11: ÉVALUATION RAPIDE
# ============================================================

print("\n" + "="*60)
print("ÉVALUATION RAPIDE V3")
print("="*60)

FastVisionModel.for_inference(model)

def generate_response(image_path: str, question: str, max_new_tokens: int = 256):
    """Génère une réponse."""
    try:
        image = Image.open(image_path).convert('RGB').resize(TARGET_IMAGE_SIZE)
    except:
        return "Erreur"
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ]
        }
    ]
    
    input_text = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = processor(
        text=[input_text],
        images=[[image]],
        return_tensors="pt",
        padding=True,
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    
    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.lower().split("assistant")[-1].strip()
    return response

def extract_disease(response: str) -> str:
    """Extrait la maladie de la réponse."""
    response_lower = response.lower()
    if "cmd" in response_lower or "mosaïque" in response_lower or "mosaic" in response_lower:
        return "CMD"
    elif "cbsd" in response_lower or "striure" in response_lower or "brown streak" in response_lower:
        return "CBSD"
    elif "cbb" in response_lower or "bactériose" in response_lower or "bacterial" in response_lower:
        return "CBB"
    elif "cgm" in response_lower or "acarien" in response_lower or "green" in response_lower:
        return "CGM"
    elif "sain" in response_lower or "healthy" in response_lower:
        return "Healthy"
    return "Unknown"

# Évaluer sur validation set
identification_question = "Quelle maladie affecte cette feuille de manioc ?"

print("\n📊 Évaluation sur 100 samples...")
eval_samples = random.sample(val_formatted, min(100, len(val_formatted)))

results = []
for sample in tqdm(eval_samples, desc="Evaluating"):
    response = generate_response(sample['image_path'], identification_question)
    predicted = extract_disease(response)
    actual = sample['class_name']
    results.append({
        'actual': actual,
        'predicted': predicted,
        'correct': predicted == actual
    })

correct = sum(1 for r in results if r['correct'])
total = len(results)
global_accuracy = correct / total * 100

print(f"\n" + "="*60)
print(f"RÉSULTATS V3 vs Stage 1 vs V2")
print(f"="*60)
print(f"\n📊 Accuracy globale:")
print(f"  - Stage 1: {STAGE1_RESULTS['global']:.1f}%")
print(f"  - Stage 2 V2: 31.0%")
print(f"  - Stage 2 V3: {global_accuracy:.1f}%")

# Par classe
print("\n📈 Par classe (V3 vs Stage 1):")
class_metrics = {}
for cls in CLASS_NAMES:
    cls_samples = [r for r in results if r['actual'] == cls]
    if cls_samples:
        cls_correct = sum(1 for r in cls_samples if r['correct'])
        acc = cls_correct / len(cls_samples) * 100
        class_metrics[cls] = acc
        stage1_acc = STAGE1_RESULTS.get(cls, 0)
        diff = acc - stage1_acc
        status = "✅" if diff >= -10 else "⚠️" if diff >= -20 else "❌"
        print(f"  {status} {CLASS_FULL_NAMES[cls]}: {acc:.1f}% (Stage 1: {stage1_acc:.1f}%, Δ{diff:+.1f}%)")


ÉVALUATION RAPIDE V3

📊 Évaluation sur 100 samples...


Evaluating: 100%|██████████| 100/100 [09:49<00:00,  5.89s/it]


RÉSULTATS V3 vs Stage 1 vs V2

📊 Accuracy globale:
  - Stage 1: 80.5%
  - Stage 2 V2: 31.0%
  - Stage 2 V3: 1.0%

📈 Par classe (V3 vs Stage 1):
  ❌ Bactériose du manioc (CBB): 0.0% (Stage 1: 55.1%, Δ-55.1%)
  ❌ Striure brune du manioc (CBSD): 0.0% (Stage 1: 84.0%, Δ-84.0%)
  ❌ Acarien vert du manioc (CGM): 5.6% (Stage 1: 56.5%, Δ-50.9%)
  ❌ Mosaïque du manioc (CMD): 0.0% (Stage 1: 86.0%, Δ-86.0%)
  ❌ Sain: 0.0% (Stage 1: 75.4%, Δ-75.4%)


In [12]:
# ============================================================
# CELL 12: SAUVEGARDE RÉSULTATS
# ============================================================

eval_results = {
    "stage": "2_v3_from_stage1",
    "total_samples": len(results),
    "global_accuracy_v3": global_accuracy,
    "global_accuracy_stage1": STAGE1_RESULTS['global'],
    "global_accuracy_v2": 31.0,
    "class_metrics_v3": class_metrics,
    "class_metrics_stage1": STAGE1_RESULTS,
    "improvement_vs_v2": global_accuracy - 31.0,
    "retention_vs_stage1": global_accuracy - STAGE1_RESULTS['global'],
}

with open(CHECKPOINT_DIR / "eval_results.json", 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"\n✅ Résultats sauvegardés")
print(f"\n" + "="*60)
print("CONCLUSION")
print("="*60)

retention = global_accuracy - STAGE1_RESULTS['global']
improvement_v2 = global_accuracy - 31.0

if retention >= -10:
    print(f"\n🎉 Rétention excellente! ({retention:+.1f}% vs Stage 1)")
elif retention >= -20:
    print(f"\n⚠️ Rétention acceptable ({retention:+.1f}% vs Stage 1)")
else:
    print(f"\n❌ Rétention insuffisante ({retention:+.1f}% vs Stage 1)")

if improvement_v2 > 0:
    print(f"📈 Amélioration vs V2: +{improvement_v2:.1f}%")

print(f"\nCheckpoint: {CHECKPOINT_DIR / 'final_model'}")


✅ Résultats sauvegardés

CONCLUSION

❌ Rétention insuffisante (-79.5% vs Stage 1)

Checkpoint: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_v3_from_stage1/final_model


In [13]:
# ============================================================
# CELL 13: DIAGNOSTIC - VOIR LES RÉPONSES BRUTES
# ============================================================

print("=" * 60)
print("DIAGNOSTIC: RÉPONSES BRUTES DU MODÈLE")
print("=" * 60)

# Prendre 10 samples pour diagnostic
diagnostic_samples = eval_samples[:10]

for i, sample in enumerate(diagnostic_samples):
    response = generate_response(sample['image_path'], identification_question)
    predicted = extract_disease(response)
    
    print(f"\n{'='*60}")
    print(f"[{i+1}] Classe réelle: {sample['class_name']}")
    print(f"    Classe prédite: {predicted}")
    print(f"    Image: {Path(sample['image_path']).name}")
    print(f"{'='*60}")
    print(f"RÉPONSE BRUTE:")
    print(f"{response[:500]}")
    if len(response) > 500:
        print("...")

DIAGNOSTIC: RÉPONSES BRUTES DU MODÈLE

[1] Classe réelle: Healthy
    Classe prédite: Unknown
    Image: 1047958369.jpg
RÉPONSE BRUTE:
la feuille de manioc dans l'image semble être en bonne santé, sans signes visibles de maladies ou de dommages. cependant, il est important de noter que la détermination précise d'une maladie nécessite une inspection plus approfondie et parfois des tests de laboratoire. les symptômes d'une maladie sur une plante peuvent varier considérablement selon le type de maladie et sa localisation.

si vous avez des préoccupations concernant la santé de votre plante, je vous recommande de consulter un profe
...

[2] Classe réelle: CMD
    Classe prédite: Unknown
    Image: uuid_9bf19e0c-dd20-4698-a160-e023b0fd2dd4.jpg
RÉPONSE BRUTE:
la feuille de manioc dans l'image semble avoir des taches et des déformations, ce qui pourrait être dû à diverses causes. voici quelques hypothèses possibles :

1. **maladie foliage** : certaines maladies bactériennes ou fongiques peuve